##LAB 1: NON-STREAMING PIPELINE

In [ ]:
# Install required packages
print("📦 Installing required packages...\n")
!pip install -q transformers datasets torch
print("\n✅ Installation complete!")

📦 Installing required packages...


✅ Installation complete!


In [ ]:
# Import libraries
print("📚 Importing libraries...\n")

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch
import time
from datetime import datetime

print("✅ All imports successful!")
print(f"\n🔍 PyTorch version: {torch.__version__}")
print(f"🔍 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔍 GPU: {torch.cuda.get_device_name(0)}")

📚 Importing libraries...

✅ All imports successful!

🔍 PyTorch version: 2.8.0+cu126
🔍 CUDA available: True
🔍 GPU: Tesla T4


In [ ]:
print("="*60)
print("STEP 1: LOADING DATASET")
print("="*60)
print(f"\n⏰ Started at: {datetime.now().strftime('%H:%M:%S')}")
print("\n📥 Downloading Yelp Reviews dataset...")
print("   (First time may take 2-3 minutes)\n")

start_time = time.time()

# Load Yelp Reviews dataset (650K restaurant reviews)
dataset = load_dataset("yelp_review_full", split="train")

# Select subset for memory efficiency
print("✅ Dataset downloaded!")
print(f"   Total reviews: {len(dataset):,}")
print("\n🔪 Selecting first 100,000 reviews...")
dataset = dataset.select(range(100000))

load_time = time.time() - start_time

print("\n" + "="*60)
print("✅ DATASET LOADED SUCCESSFULLY")
print("="*60)
print(f"📊 Number of reviews: {len(dataset):,}")
print(f"⏱️  Loading time: {load_time:.2f} seconds")
print(f"💾 Approximate memory: ~2-3 GB")

STEP 1: LOADING DATASET

⏰ Started at: 17:30:26

📥 Downloading Yelp Reviews dataset...
   (First time may take 2-3 minutes)



README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

✅ Dataset downloaded!
   Total reviews: 650,000

🔪 Selecting first 100,000 reviews...

✅ DATASET LOADED SUCCESSFULLY
📊 Number of reviews: 100,000
⏱️  Loading time: 13.58 seconds
💾 Approximate memory: ~2-3 GB


In [ ]:
# Preview the data
print("\n" + "="*60)
print("📝 SAMPLE REVIEWS")
print("="*60)

for i in range(3):
    print(f"\n{'─'*60}")
    print(f"Review {i}: {'⭐' * dataset[i]['label']} stars")
    print(f"{'─'*60}")
    print(dataset[i]['text'][:300])
    print("...")

# Calculate average correctly
print(f"\n Average review length: {sum(len(dataset[i]['text']) for i in range(100))/100:.0f} characters")


📝 SAMPLE REVIEWS

────────────────────────────────────────────────────────────
Review 0: ⭐⭐⭐⭐ stars
────────────────────────────────────────────────────────────
dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens
...

────────────────────────────────────────────────────────────
Review 1: ⭐ stars
────────────────────────────────────────────────────────────
Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has ti
...

────────────────────────────────────────────────────────────
Review 2: ⭐⭐⭐ stars
─────

##Initialize Tokenizer

In [ ]:
print("="*60)
print("STEP 2: TOKENIZER SETUP")
print("="*60)
print("\n🔧 Loading GPT-2 tokenizer...\n")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set pad token

print("✅ Tokenizer loaded successfully!")
print(f"\n Vocabulary size: {tokenizer.vocab_size:,} tokens")
print(f"🔑 EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"🔑 PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

STEP 2: TOKENIZER SETUP

🔧 Loading GPT-2 tokenizer...



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Tokenizer loaded successfully!

 Vocabulary size: 50,257 tokens
🔑 EOS token: '<|endoftext|>' (ID: 50256)
🔑 PAD token: '<|endoftext|>' (ID: 50256)


In [ ]:
# Test tokenizer with a sample review
print("\n" + "="*60)
print(" TOKENIZER TEST")
print("="*60)

sample_text = "The food was amazing! Best restaurant I've been to."
sample_tokens = tokenizer(sample_text)

print(f"\n📝 Input text: '{sample_text}'")
print(f"\n🔢 Token IDs: {sample_tokens['input_ids']}")
print(f"\n🔄 Decoded back: '{tokenizer.decode(sample_tokens['input_ids'])}'")
print(f"\n📏 Number of tokens: {len(sample_tokens['input_ids'])}")


 TOKENIZER TEST

📝 Input text: 'The food was amazing! Best restaurant I've been to.'

🔢 Token IDs: [464, 2057, 373, 4998, 0, 6705, 7072, 314, 1053, 587, 284, 13]

🔄 Decoded back: 'The food was amazing! Best restaurant I've been to.'

📏 Number of tokens: 12


In [ ]:
print("="*60)
print("STEP 3: TOKENIZING DATASET")
print("="*60)
print(f"\n⏰ Started at: {datetime.now().strftime('%H:%M:%S')}")
print("\n⚙️  Tokenizing 100,000 reviews...")
print("   (This will take 3-5 minutes)\n")

start_time = time.time()

def tokenize_function(examples):
    """
    Tokenize text without adding special tokens or padding
    """
    return tokenizer(examples["text"], return_special_tokens_mask=False)

# Apply tokenization to entire dataset
tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,               # Process in batches for speed
    remove_columns=["text", "label"],  # Remove original columns
    desc="Tokenizing",          # Progress bar description
    num_proc=4                  # Use multiple CPU cores
)

tokenize_time = time.time() - start_time

print("\n" + "="*60)
print("✅ TOKENIZATION COMPLETE")
print("="*60)
print(f"⏱️  Time taken: {tokenize_time:.2f} seconds ({tokenize_time/60:.2f} minutes)")
print(f"📊 Reviews tokenized: {len(tokenized_ds):,}")
print(f"⚡ Speed: {len(tokenized_ds)/tokenize_time:.2f} reviews/second")

STEP 3: TOKENIZING DATASET

⏰ Started at: 17:37:32

⚙️  Tokenizing 100,000 reviews...
   (This will take 3-5 minutes)



Tokenizing (num_proc=4):   0%|          | 0/100000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1143 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1225 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1057 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1132 > 1024). Running this sequence through the model will result in indexing errors



✅ TOKENIZATION COMPLETE
⏱️  Time taken: 48.29 seconds (0.80 minutes)
📊 Reviews tokenized: 100,000
⚡ Speed: 2070.88 reviews/second


In [ ]:
# Inspect tokenized data
print("\n" + "="*60)
print(" TOKENIZED DATA PREVIEW")
print("="*60)

sample_tokens = tokenized_ds[0]['input_ids']

print(f"\n📊 First review statistics:")
print(f"   - Number of tokens: {len(sample_tokens)}")
print(f"   - First 20 tokens: {sample_tokens[:20]}")
print(f"\n📝 Decoded text (first 200 chars):")
print(f"   {tokenizer.decode(sample_tokens)[:200]}...")


 TOKENIZED DATA PREVIEW

📊 First review statistics:
   - Number of tokens: 119
   - First 20 tokens: [7109, 13, 3869, 3900, 4394, 2279, 1312, 804, 329, 287, 257, 2276, 32110, 13, 220, 339, 338, 3621, 290, 2562]

📝 Decoded text (first 200 chars):
   dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-no...


In [ ]:
print("="*60)
print("STEP 4: GROUPING INTO FIXED-LENGTH BLOCKS")
print("="*60)
print(f"\n⏰ Started at: {datetime.now().strftime('%H:%M:%S')}")

block_size = 128  # Sequence length for training

print(f"\n🎯 Target block size: {block_size} tokens")
print(f"\n⚙️  Processing...\n")

start_time = time.time()

def group_texts(examples):
    """
    Concatenate all texts and split into blocks of block_size.

    This is the standard approach for language model training:
    - Concatenate across document boundaries
    - Create fixed-length sequences
    - Maximize token usage
    """
    # Concatenate all input_ids and attention_masks
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks = sum(examples["attention_mask"], [])

    # Calculate total usable length (multiple of block_size)
    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks = concatenated_masks[:total_len]

    # Split into chunks of block_size
    result_input_ids = [
        concatenated_inputs[i:i+block_size]
        for i in range(0, total_len, block_size)
    ]
    result_masks = [
        concatenated_masks[i:i+block_size]
        for i in range(0, total_len, block_size)
    ]

    return {
        "input_ids": result_input_ids,
        "attention_mask": result_masks
    }

# Apply grouping
lm_ds = tokenized_ds.map(
    group_texts,
    batched=True,
    batch_size=1000,
    desc="Grouping texts"
)

group_time = time.time() - start_time

print("\n" + "="*60)
print("✅ GROUPING COMPLETE")
print("="*60)
print(f"📊 Training sequences created: {len(lm_ds):,}")
print(f"📏 Sequence length: {block_size} tokens")
print(f"⏱️  Time taken: {group_time:.2f} seconds")
print(f"\n💡 From 100K reviews → {len(lm_ds):,} training sequences")

STEP 4: GROUPING INTO FIXED-LENGTH BLOCKS

⏰ Started at: 17:39:08

🎯 Target block size: 128 tokens

⚙️  Processing...



Grouping texts:   0%|          | 0/100000 [00:00<?, ? examples/s]


✅ GROUPING COMPLETE
📊 Training sequences created: 129,089
📏 Sequence length: 128 tokens
⏱️  Time taken: 95.64 seconds

💡 From 100K reviews → 129,089 training sequences


In [ ]:
# Verify block structure
print("\n" + "="*60)
print("🔍 VERIFYING BLOCK STRUCTURE")
print("="*60)

sample_block = lm_ds[0]

print(f"\n✅ Sample block (sequence 0):")
print(f"   - input_ids length: {len(sample_block['input_ids'])}")
print(f"   - attention_mask length: {len(sample_block['attention_mask'])}")
print(f"\n📝 Decoded text (first 200 chars):")
print(f"   {tokenizer.decode(sample_block['input_ids'])[:200]}...")


🔍 VERIFYING BLOCK STRUCTURE

✅ Sample block (sequence 0):
   - input_ids length: 128
   - attention_mask length: 128

📝 Decoded text (first 200 chars):
   dr. goldberg offers everything i look for in a general practitioner.  he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-no...


In [ ]:
print("="*60)
print("STEP 5: CREATING DATALOADER")
print("="*60)

def collate_fn(batch):
    """
    Collate function for DataLoader.

    For causal language modeling:
    - input_ids: the token sequences
    - labels: same as input_ids (model learns to predict next token)
    """
    input_ids = torch.tensor(
        [example["input_ids"] for example in batch],
        dtype=torch.long
    )
    return {
        "input_ids": input_ids,
        "labels": input_ids.clone()  # Labels are input_ids for LM
    }

# Create DataLoader
batch_size = 8
train_loader = DataLoader(
    lm_ds,
    batch_size=batch_size,
    shuffle=True,              # Shuffle for randomness
    collate_fn=collate_fn
)

print(f"\n✅ DataLoader created successfully!")
print(f"\n📊 Configuration:")
print(f"   - Batch size: {batch_size}")
print(f"   - Total batches: {len(train_loader):,}")
print(f"   - Shuffle: Enabled")
print(f"\n💾 Each batch shape: ({batch_size}, {block_size})")

STEP 5: CREATING DATALOADER

✅ DataLoader created successfully!

📊 Configuration:
   - Batch size: 8
   - Total batches: 16,137
   - Shuffle: Enabled

💾 Each batch shape: (8, 128)


In [ ]:
print("="*60)
print("STEP 6: INSPECTING SAMPLE BATCHES")
print("="*60)

print("\n🔍 Fetching 3 sample batches...\n")

for i, batch in enumerate(train_loader):
    print(f"\n{'─'*60}")
    print(f"📦 Batch {i}:")
    print(f"{'─'*60}")
    print(f"   input_ids shape:  {batch['input_ids'].shape}")
    print(f"   labels shape:     {batch['labels'].shape}")
    print(f"   dtype:            {batch['input_ids'].dtype}")

    # Show decoded text for first batch
    if i == 0:
        print(f"\n   📝 First sequence (decoded):")
        decoded = tokenizer.decode(batch['input_ids'][0])
        print(f"   {decoded[:300]}...")

    if i == 2:  # Show 3 batches total
        break

print(f"\n\n{'='*60}")
print("✅ ALL BATCHES VERIFIED SUCCESSFULLY")
print(f"{'='*60}")

STEP 6: INSPECTING SAMPLE BATCHES

🔍 Fetching 3 sample batches...


────────────────────────────────────────────────────────────
📦 Batch 0:
────────────────────────────────────────────────────────────
   input_ids shape:  torch.Size([8, 128])
   labels shape:     torch.Size([8, 128])
   dtype:            torch.int64

   📝 First sequence (decoded):
    our group an area of big horn sheep coming down a mountain and into a residential park area which was fantastic.  Our group took a short boat ride on the river below the Hoover Dam that provided us with a different perspective and knowledge of the Dam that not everyone is able to experience.  The t...

────────────────────────────────────────────────────────────
📦 Batch 1:
────────────────────────────────────────────────────────────
   input_ids shape:  torch.Size([8, 128])
   labels shape:     torch.Size([8, 128])
   dtype:            torch.int64

────────────────────────────────────────────────────────────
📦 Batch 2:
───────────────────

In [ ]:
print("\n\n" + "="*60)
print("🎉 LAB 1 COMPLETE - SUMMARY")
print("="*60)

print(f"\n📊 DATASET STATISTICS:")
print(f"   {'─'*50}")
print(f"   Dataset:              Yelp Reviews")
print(f"   Reviews loaded:       {100000:,}")
print(f"   Training sequences:   {len(lm_ds):,}")
print(f"   Sequence length:      {block_size} tokens")
print(f"   Vocabulary size:      {tokenizer.vocab_size:,}")

print(f"\n⚙️  PIPELINE CONFIGURATION:")
print(f"   {'─'*50}")
print(f"   Model tokenizer:      GPT-2")
print(f"   Batch size:           {batch_size}")
print(f"   Total batches:        {len(train_loader):,}")
print(f"   Shuffle:              Enabled")

print(f"\n💾 MEMORY CHARACTERISTICS:")
print(f"   {'─'*50}")
print(f"   Approach:             Non-streaming (in-memory)")
print(f"   Estimated RAM:        ~2-3 GB")
print(f"   Max dataset size:     Limited by available RAM")

print(f"\n🎯 READY FOR TRAINING:")
print(f"   {'─'*50}")
print(f"   ✅ Data pipeline is complete")
print(f"   ✅ DataLoader ready for model training")
print(f"   ✅ All batches verified")




🎉 LAB 1 COMPLETE - SUMMARY

📊 DATASET STATISTICS:
   ──────────────────────────────────────────────────
   Dataset:              Yelp Reviews
   Reviews loaded:       100,000
   Training sequences:   129,089
   Sequence length:      128 tokens
   Vocabulary size:      50,257

⚙️  PIPELINE CONFIGURATION:
   ──────────────────────────────────────────────────
   Model tokenizer:      GPT-2
   Batch size:           8
   Total batches:        16,137
   Shuffle:              Enabled

💾 MEMORY CHARACTERISTICS:
   ──────────────────────────────────────────────────
   Approach:             Non-streaming (in-memory)
   Estimated RAM:        ~2-3 GB
   Max dataset size:     Limited by available RAM

🎯 READY FOR TRAINING:
   ──────────────────────────────────────────────────
   ✅ Data pipeline is complete
   ✅ DataLoader ready for model training
   ✅ All batches verified
